In [1]:
import sys
import os

sys.path.insert(0, os.path.abspath("/home/ElasticNotebook"))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/new_notebooks/nyc-flight/annotated/checkpoints/post_cell_9.pickle

trying: ['df', 'orig_output']
me:  10
me:  13
trying: ['df', 'orig_output']
me:  10
me:  13
trying: ['pd']
me:  0
trying: ['f_total']
me:  8
trying: ['f']
me:  8
trying: ['sklearn']
me:  0
trying: ['flights_df']
me:  10
trying: ['df2']
me:  10
trying: ['np']
me:  0
trying: ['time']
me:  0
trying: ['matplotlib']
me:  0
trying: ['plt']
me:  0
trying: ['sp']
me:  0
trying: ['IPython']
me:  0
Declaring variable pd
Declaring variable sklearn
Declaring variable np
Declaring variable time
Declaring variable matplotlib
Declaring variable plt
Declaring variable sp
Declaring variable IPython
Declaring variable f_total
Declaring variable f
Declaring variable df
Declaring variable df
Declaring variable flights_df
Declaring variable df2
Declaring variable orig_output
Declaring variable orig_output


In [4]:
%%RecordEvent
%%time
### cell 10 ###
# Use cudf’s native mean aggregator (by name) rather than NumPy to stay on GPU
# and preserve identical semantics to the original .agg(np.mean)
df = flights_df.groupby(
    ['day', 'month'],
    as_index=False
).agg({
    'arr_delay': 'mean',
    'dep_delay': 'mean'
})
# compute the combined delay
df['total_delay'] = df['arr_delay'] + df['dep_delay']
# show the (day, month) with the highest total delay
df.sort_values('total_delay', ascending=False).head(1)

CPU times: user 34.3 ms, sys: 23.6 ms, total: 57.9 ms
Wall time: 57.9 ms


,day,month,arr_delay,dep_delay,total_delay
196,17,5,1.093878,7.933673,9.027551


In [5]:
%Checkpoint /home/dias-benchmarks/new_notebooks/nyc-flight/rewritten/o4_mini_high/checkpoints/post_cell_10_try_3.pickle

migration speed (bps): 749790636.4948379
---------------------------
variables to migrate:
flights_df 48
df 48
sklearn 72
time 72
pd 72
matplotlib 72
np 72
sp 72
plt 72
f 48
df2 48
f_total 28
orig_output 48
IPython 72
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/new_notebooks/nyc-flight/rewritten/o4_mini_high/checkpoints/post_cell_10_try_3.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['flights_df']
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables []
Intermediate variables []
Future variables ['flights_df']
Modified dataframes
======= Cell 2 =======
Input variables []
Active variables []
Intermediate variables []
Future variables ['flights_df']
Modified dataframes
======= Cell 3 =======
Input variables []
Active variables []
Intermediate variables []
Future variables ['flights_df']
Modified dataframes
======= Cell 4 =======
Input variables []
Active variables []
Intermediate variables []
Future variables ['flights_df']
Modified dataframes
======= Cell 5 =======
Input variables []
Active variables []
Intermediate variables []
Future variables ['flights_df']
Modified dataframes
======= Cell 6 =======
Input variables []
Active variables []
Intermediate variables []
Future variables ['flights_df']
Modified dataframes
======= Cell 7 =====

In [7]:
with open(
    "/home/dias-benchmarks/new_notebooks/nyc-flight/opt_cell_exec_info_10_try_3.pkl",
    "wb",
) as f:
    pickle.dump(opt_cell_exec_info[10], f)

In [ ]:
opt_output = Out.get(4)

In [ ]:
df_opt = df
%LoadCheckpoint /home/dias-benchmarks/new_notebooks/nyc-flight/annotated/checkpoints/post_cell_10.pickle
assert compare_df(df_opt, df)

import numpy as np
from elastic.core.common.pandas import is_type_styler

is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(
    orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray)
)
is_opt_output_array = isinstance(
    opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray)
)
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif (
    (is_orig_output_pd or is_opt_output_pd)
    and (is_orig_output_array or is_opt_output_array)
) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output